# Tutorial 5: Train NicheTrans on STARmap PLUS data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_STARmap_PLUS import AD_Mouse

from utils.utils import *
from utils.utils_training_STARmap_PLUS import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.25, n_top_genes=2000, workers=4, AD_adata_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nn_AD_mouse/AD_model_adata_protein', Wild_type_adata_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nn_AD_mouse/wild_type_adata_protein', max_epoch=20, stepsize=20, train_batch=128, test_batch=32, optimizer='adam', lr=0.0001, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = AD_Mouse(AD_adata_path=args.AD_adata_path, Wild_type_adata_path=args.Wild_type_adata_path, n_top_genes=args.n_top_genes)
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.target_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate)
model = nn.DataParallel(model).cuda()

------Calculating spatial graph...
The graph contains 124464 edges, 10372 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 115608 edges, 9634 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 96408 edges, 8034 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  10372 spots, 894.0 positive tao, 291.0 positive plaque 
  test     |   9634 spots, 620.0 positive tao, 195.0 positive plaque 
  ------------------------------


### Load bio-prior

In [4]:
from prior_AddOn.gene_embedding_loader import load_static_gene_prior
priors = load_static_gene_prior(
    source_panel=dataset.source_panel,
    models=("scgpt",),
    species="human",  
    root="prior_AddOn/gene_embeddings",
)


### Initialize loss function (criterion) and optimizer

In [5]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [6]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader)
torch.save(model.state_dict(), 'NicheTrans_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 81/81	 Loss 0.587726 (0.678430)
==> Epoch 2/20
Batch 81/81	 Loss 0.456607 (0.516254)
==> Epoch 3/20
Batch 81/81	 Loss 0.364311 (0.406195)
==> Epoch 4/20
Batch 81/81	 Loss 0.297478 (0.331293)
==> Epoch 5/20
Batch 81/81	 Loss 0.240850 (0.278405)
==> Epoch 6/20
Batch 81/81	 Loss 0.225955 (0.238072)
==> Epoch 7/20
Batch 81/81	 Loss 0.207906 (0.209849)
==> Epoch 8/20
Batch 81/81	 Loss 0.140652 (0.187564)
==> Epoch 9/20
Batch 81/81	 Loss 0.183276 (0.171454)
==> Epoch 10/20
Batch 81/81	 Loss 0.179350 (0.160803)
==> Epoch 11/20
Batch 81/81	 Loss 0.141747 (0.149167)
==> Epoch 12/20
Batch 81/81	 Loss 0.149339 (0.142047)
==> Epoch 13/20
Batch 81/81	 Loss 0.137734 (0.132499)
==> Epoch 14/20
Batch 81/81	 Loss 0.167671 (0.128104)
==> Epoch 15/20
Batch 81/81	 Loss 0.102368 (0.125024)
==> Epoch 16/20
Batch 81/81	 Loss 0.123153 (0.119902)
==> Epoch 17/20
Batch 81/81	 Loss 0.139399 (0.118476)
==> Epoch 18/20
Batch 81/81	 Loss 0.110434 (0.114503)
==> Epoch 19/20
Batch 81/81	 Loss 0.0

In [9]:
for row in priors["scgpt"]["mapping_table"]:
    if row["status"] != "mapped":
        print(row["input_gene"], row["status"], row["reason"])

Akr1c18 missing_embedding human_symbol_not_in_scgpt_vocab
Ank missing_embedding human_symbol_not_in_scgpt_vocab
Car8 missing_embedding human_symbol_not_in_scgpt_vocab
Cbr2 missing_embedding human_symbol_not_in_scgpt_vocab
Ccl12 missing_embedding human_symbol_not_in_scgpt_vocab
Ccl6 missing_embedding human_symbol_not_in_scgpt_vocab
Ccl9 missing_embedding human_symbol_not_in_scgpt_vocab
Ceacam10 missing_embedding human_symbol_not_in_scgpt_vocab
Clca3a1 missing_embedding human_symbol_not_in_scgpt_vocab
Ctla2a missing_embedding human_symbol_not_in_scgpt_vocab
Dlx1as missing_embedding human_symbol_not_in_scgpt_vocab
E330013p04rik missing_embedding human_symbol_not_in_scgpt_vocab
Evx1os missing_embedding human_symbol_not_in_scgpt_vocab
Fcrls missing_embedding human_symbol_not_in_scgpt_vocab
Gkn3 missing_embedding human_symbol_not_in_scgpt_vocab
H2-aa missing_embedding human_symbol_not_in_scgpt_vocab
H2-ab1 missing_embedding human_symbol_not_in_scgpt_vocab
H2-dmb2 missing_embedding human_symb